# Notebook C: kiểm tra bổ sung cho phép thử TLD chưa thấy

**Đề tài:** Một quy tắc một dòng so với mười mô hình trên bộ dữ liệu lừa đảo trực tuyến phổ biến và độ chính xác gần tuyệt đối thật ra đo điều gì

Notebook B chạy phép thử TLD chưa thấy trên một lần chia (`random_state=0`). Notebook này lặp lại phép thử trên **năm lần chia khác nhau** (`random_state` từ 0 đến 4) cho cả mười mô hình và 21 đặc trưng địa chỉ, kèm control cùng kích thước, ở bốn mức ngân sách báo động giả (0,5; 1; 2 và 5%).

Chạy sau Notebook B; chỉ cần `data/processed/phishing.parquet` và `sql/queries.sql`. Notebook này không sửa kết quả nào của Notebook A và B, chỉ ghi thêm năm file trong `report/`: `table_review_tld_splits.csv`, `table_review_tld_splits_info.csv`, `table_review_tld_splits_summary.csv`, `table_review_fa_budget.csv` và `table_review_fa_budget_models.csv`. Lần chia 0 chính là lần chia của Notebook B nên phải tái tạo được `table_rq3_unseen_tld.csv`; ô cuối kiểm tra điều này.

In [1]:
# run once per machine, then comment out
# %pip install pandas numpy duckdb pyarrow scikit-learn lightgbm xgboost matplotlib scipy tldextract
import os, json, hashlib, numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
import glob
def project_root(notebook='Notebook_C_OnGoingAssessment2_ACDSA_PhishingURL_OneLineRule.ipynb', need='data/processed/manifest.json'):          # the folder that holds this notebook and data/processed/manifest.json, wherever Jupyter or VS Code starts
    here = globals().get('__vsc_ipynb_file__'); starts = ([os.path.dirname(here)] if here else []) + [os.getcwd()]
    cands = []
    for s in starts:
        cands += [s, os.path.dirname(s)] + sorted(glob.glob(os.path.join(s, '*'))) + sorted(glob.glob(os.path.join(s, '*', '*')))
    cands = [c for c in dict.fromkeys(os.path.abspath(c) for c in cands) if os.path.isfile(os.path.join(c, notebook))]
    best = [c for c in cands if os.path.exists(os.path.join(c, need))]
    return (best or cands or [os.getcwd()])[0]
os.chdir(project_root())
if os.path.exists('data/processed/manifest.json'): print('working folder:', os.path.basename(os.getcwd()), '| data/processed/manifest.json present: True')
else: print('working folder:', os.getcwd(), '| data/processed/manifest.json NOT found here: unzip the whole project (with its data folder) and open this notebook from that folder')
m = json.load(open('data/processed/manifest.json'))
assert hashlib.md5(open(m['file'], 'rb').read()).hexdigest() == m['md5'], 'parquet differs from the manifest, rerun Notebook A'
os.makedirs('report', exist_ok=True); print('handover ok', m['rows'], 'rows')

working folder: OnGoingAssessment2_ACDSA_PhishingURL_OneLineRule | data/processed/manifest.json present: True
handover ok 235370 rows


## Bước 1a. Bảng đặc trưng và nhóm đặc trưng

- **Làm gì:** chạy lại `sql/queries.sql`, lấy bảng `feat` và xác định 21 đặc trưng địa chỉ giống hệt Notebook B.
- **Vì sao:** năm lần chia phải dùng cùng một bảng và cùng một tập đặc trưng với phép thử gốc.
- **Kiểm tra:** dòng `assert len(ADDRESS) == 21` qua; ô in 235.370 URL và 21 đặc trưng địa chỉ.

In [2]:
# Step 1a: the same feature table, feature groups and target as Notebook B
import duckdb
from sklearn.model_selection import train_test_split
con = duckdb.connect(config={'threads': 2, 'memory_limit': '1GB'}); con.execute(open('sql/queries.sql').read())
try:
    feat = con.execute("SELECT * FROM feat ORDER BY row_id").df()
except duckdb.OutOfMemoryException:
    feat = pd.read_parquet('data/processed/phishing.parquet').drop(columns=['URL']).sort_values('row_id').reset_index(drop=True)
SHORTCUT = ['URLSimilarityIndex']
ADDRESS = [c for c in ['URLLength', 'DomainLength', 'IsDomainIP', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation',
           'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL',
           'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS'] if c in feat.columns]
TARGET = 'phishing'; assert len(ADDRESS) == 21
print(len(feat), 'URLs,', len(ADDRESS), 'address features')

235370 URLs, 21 address features


## Bước 1b. Mười mô hình

- **Làm gì:** định nghĩa lại mười mô hình và hàm `proba` đúng như bước 6a của Notebook B; ô này chưa huấn luyện gì.
- **Vì sao:** cùng siêu tham số với Notebook B thì kết quả lần chia 0 mới so sánh được.
- **Kiểm tra:** ô chạy không lỗi; siêu tham số không đổi so với Notebook B.

In [8]:
# Step 1b: ten classifiers from four families, five seeds, same stratified split, phishing against legitimate addresses; probability metrics because one class is rare
import numpy as np, pandas as pd, time, gc
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
sc = lambda m: make_pipeline(StandardScaler(), m)       # scaling for linear, distance and neural models
models = {
    'Logistic':       lambda s: sc(LogisticRegression(max_iter=500, random_state=s)),
    'Linear SVM':     lambda s: sc(LinearSVC(C=1.0, random_state=s)),
    'k-NN':           lambda s: sc(KNeighborsClassifier(n_neighbors=15)),
    'Decision Tree':  lambda s: DecisionTreeClassifier(max_depth=12, min_samples_leaf=5, random_state=s),
    'Random Forest':  lambda s: RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=s),
    'Extra Trees':    lambda s: ExtraTreesClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=s),
    'HistGB':         lambda s: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, random_state=s),
    'LightGBM':       lambda s: LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, subsample=0.8,
                                               subsample_freq=1, colsample_bytree=0.8, random_state=s, verbose=-1),
    'XGBoost':        lambda s: XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8,
                                              colsample_bytree=0.8, tree_method='hist', random_state=s),
    'Small MLP':      lambda s: sc(MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=200, early_stopping=True,
                                                 random_state=s)),
}
def proba(m, X):                                        # probability of the positive class for every model family
    if hasattr(m, 'predict_proba'): return m.predict_proba(X)[:, 1]
    return 1 / (1 + np.exp(-m.decision_function(X)))
FAM = list(models)
print(len(models), 'models:', ', '.join(FAM))

10 models: Logistic, Linear SVM, k-NN, Decision Tree, Random Forest, Extra Trees, HistGB, LightGBM, XGBoost, Small MLP


## Bước 2. Phép thử TLD chưa thấy trên năm lần chia

- **Làm gì:** với mỗi lần chia s từ 0 đến 4, lấy mẫu 100.000 URL bằng `random_state=s`, chia 60/20/20, rồi huấn luyện mười mô hình chỉ trên .com, .org và .net (hạt ngẫu nhiên của mô hình bằng s). Ngưỡng đặt trên URL hợp pháp của tập validation cùng ba TLD, tại bốn mức ngân sách 0,5; 1; 2 và 5%. Sau đó đo tỷ lệ bắt và tỷ lệ báo nhầm trên các TLD còn lại. Control huấn luyện trên mẫu ngẫu nhiên cùng kích thước lấy từ mọi TLD. Luật tiền tố cũng được chấm trên các TLD held-out của từng lần chia.
- **Vì sao:** một lần chia có thể may hoặc rủi; lặp năm lần cho biết kết quả có vững không. Control loại trừ khả năng do tập huấn luyện nhỏ hơn.
- **Kiểm tra:** mỗi lần chia in `split s done com org net`, tức ba TLD huấn luyện luôn là .com, .org, .net; luật tiền tố có `rule_false_alarm_unseen = 0` ở cả năm lần; ghi `table_review_tld_splits.csv` và `table_review_tld_splits_info.csv`.

In [4]:
# Step 2: the unseen-TLD test on five splits, with the size-matched control, at four false-alarm budgets
BUDGETS = [0.005, 0.01, 0.02, 0.05]
def split(s):                                           # the construction of Notebook B (cell 4) with random_state = s
    pool = feat.sample(min(len(feat), 100_000), random_state=s)
    train, rest = train_test_split(pool, test_size=0.4, random_state=s, stratify=pool[TARGET]); valid, test = train_test_split(rest, test_size=0.5, random_state=s, stratify=rest[TARGET])
    assert set(train.row_id).isdisjoint(test.row_id) and set(train.row_id).isdisjoint(valid.row_id) and set(valid.row_id).isdisjoint(test.row_id)
    return pool, train, valid, test
def evaluate(m, Vd, Bs):                                # threshold at each budget from the validation legitimate scores, then catch and false-alarm rate on every test set in Bs
    pv = proba(m, Vd.loc[Vd[TARGET] == 0, ADDRESS]); sc_ = {k: proba(m, B[ADDRESS]) for k, B in Bs.items()}; out = []
    for b in BUDGETS:
        t = np.quantile(pv, 1 - b)
        for k, B in Bs.items():
            y = B[TARGET].to_numpy(); p = sc_[k]; out.append((b, k, float((p[y == 1] >= t).mean()), float((p[y == 0] >= t).mean())))
        out.append((b, 'valid', np.nan, float((pv >= t).mean())))
    return out
rows, info = [], []
for s in range(5):
    pool, train, valid, test = split(s)
    big = list(pool.TLD.value_counts().index[:3])
    A = train[train.TLD.isin(big)]; V = valid[valid.TLD.isin(big)]; B_in = test[test.TLD.isin(big)]; B_out = test[~test.TLD.isin(big)]
    flag = 1 - B_out.starts_https_www.to_numpy(); yo = B_out[TARGET].to_numpy()          # the one-line rule on the held-out TLDs
    info.append([s, ' '.join(big), len(train), len(A), len(V), len(B_in), len(B_out), int((yo == 0).sum()), float(flag[yo == 1].mean()), float(flag[yo == 0].mean())])
    Ar, Vr = train.sample(len(A), random_state=s), valid.sample(len(V), random_state=s)      # same size as A and V, drawn from all TLDs
    for name in FAM:
        for setting, (Xa, Vd) in {'holdout': (A, V), 'control': (Ar, Vr)}.items():
            m = models[name](s).fit(Xa[ADDRESS], Xa[TARGET])
            Bs = {'out': B_out, 'in': B_in} if setting == 'holdout' else {'out': B_out}
            for b, k, cau, fa in evaluate(m, Vd, Bs):
                rows.append([s, name, setting, b, k, len(Xa), len(B_out), cau, fa])
    print('split', s, 'done', ' '.join(big), len(A), len(V), len(B_out), flush=True)
res = pd.DataFrame(rows, columns=['split', 'model', 'setting', 'budget', 'test_set', 'n_train', 'n_test_out', 'caught', 'false_alarm_rate'])
res.to_csv('report/table_review_tld_splits.csv', index=False)
inf = pd.DataFrame(info, columns=['split', 'training_tlds', 'n_train', 'n_train_tld', 'n_valid_tld', 'n_test_seen', 'n_test_unseen', 'n_legit_unseen', 'rule_caught_unseen', 'rule_false_alarm_unseen'])
inf.to_csv('report/table_review_tld_splits_info.csv', index=False); print(inf.round(4).to_string(index=False))

split 0 done com org net 35202 11799 8286
split 1 done com org net 35151 11793 8196
split 2 done com org net 35366 11715 8248
split 3 done com org net 35121 11808 8287
split 4 done com org net 35159 11758 8250
 split training_tlds  n_train  n_train_tld  n_valid_tld  n_test_seen  n_test_unseen  n_legit_unseen  rule_caught_unseen  rule_false_alarm_unseen
     0   com org net    60000        35202        11799        11714           8286            3805              0.9788                      0.0
     1   com org net    60000        35151        11793        11804           8196            3882              0.9750                      0.0
     2   com org net    60000        35366        11715        11752           8248            3895              0.9731                      0.0
     3   com org net    60000        35121        11808        11713           8287            3921              0.9753                      0.0
     4   com org net    60000        35159        11758        11

## Bước 3. Tóm tắt ở ngân sách 1%

- **Làm gì:** với mỗi mô hình, lấy tỷ lệ báo nhầm trên TLD held-out ở lần chia 0, trung bình, nhỏ nhất và lớn nhất qua năm lần chia, đặt cạnh control; tính dải cho tám mô hình có điểm liên tục (bỏ k-NN và cây quyết định vì điểm bị trùng nhau).
- **Vì sao:** đây là các con số dùng ở phần tóm tắt, mục 5.3, mục 6.1 và kết luận của bài.
- **Kiểm tra:** dòng in `holdout above control in every one of the five splits, models: 8 of 8`; ghi `table_review_tld_splits_summary.csv`.

In [5]:
# Step 3: summary at the 1% budget (the protocol of the paper): held-out-TLD false alarms against the control, per model and split
CONT = [n for n in FAM if n not in ('k-NN', 'Decision Tree')]       # the eight models with continuous scores, as in the paper
r1 = res[(res.budget == 0.01) & (res.test_set == 'out')]
wide = r1.pivot_table(index='model', columns=['setting', 'split'], values='false_alarm_rate').reindex(FAM)
summ = pd.DataFrame({'model': FAM,
    'fa_holdout_split0': wide[('holdout', 0)].values, 'fa_holdout_mean5': wide['holdout'].mean(axis=1).values, 'fa_holdout_min5': wide['holdout'].min(axis=1).values, 'fa_holdout_max5': wide['holdout'].max(axis=1).values,
    'fa_control_mean5': wide['control'].mean(axis=1).values, 'fa_control_min5': wide['control'].min(axis=1).values, 'fa_control_max5': wide['control'].max(axis=1).values})
summ['holdout_above_control_in_all_5'] = [bool((wide['holdout'].loc[n] > wide['control'].loc[n]).all()) for n in FAM]
cau = r1.pivot_table(index='model', columns='setting', values='caught', aggfunc='mean').reindex(FAM)
summ['caught_holdout_mean5'] = cau['holdout'].values; summ['caught_control_mean5'] = cau['control'].values
summ.to_csv('report/table_review_tld_splits_summary.csv', index=False); print(summ.round(4).to_string(index=False))
e = summ[summ.model.isin(CONT)]
print('eight continuous-score models, 1%% budget: holdout mean-of-five range %.2f-%.2f%%, any single split %.2f-%.2f%%; control mean-of-five range %.2f-%.2f%%, any single split %.2f-%.2f%%' % (
    100*e.fa_holdout_mean5.min(), 100*e.fa_holdout_mean5.max(), 100*e.fa_holdout_min5.min(), 100*e.fa_holdout_max5.max(), 100*e.fa_control_mean5.min(), 100*e.fa_control_mean5.max(), 100*e.fa_control_min5.min(), 100*e.fa_control_max5.max()))
print('holdout above control in every one of the five splits, models:', int(e.holdout_above_control_in_all_5.sum()), 'of', len(e))

        model  fa_holdout_split0  fa_holdout_mean5  fa_holdout_min5  fa_holdout_max5  fa_control_mean5  fa_control_min5  fa_control_max5  holdout_above_control_in_all_5  caught_holdout_mean5  caught_control_mean5
     Logistic             0.0715            0.0405           0.0227           0.0715            0.0144           0.0111           0.0205                            True                0.9946                0.9948
   Linear SVM             0.3246            0.2457           0.1798           0.3246            0.0131           0.0104           0.0180                            True                0.9974                0.9968
         k-NN             0.1714            0.1278           0.0832           0.1828            0.0249           0.0121           0.0701                            True                0.9928                0.9944
Decision Tree             0.1921            0.1786           0.0996           0.2774            0.0257           0.0218           0.0296            

## Bước 4. Các mức ngân sách báo động giả

- **Làm gì:** lặp phép tính ở bốn mức 0,5; 1; 2 và 5%. Mỗi mô hình lấy trung bình năm lần chia, rồi lấy dải giữa tám mô hình và đếm số mô hình có tỷ lệ báo nhầm vượt gấp đôi ngân sách. Bảng theo từng mô hình ghi cả mười mô hình.
- **Vì sao:** để biết kết luận có phụ thuộc vào đúng mức 1% hay không.
- **Kiểm tra:** ở validation, tỷ lệ báo nhầm xấp xỉ ngân sách (`fa_valid_min`, `fa_valid_max`); trên TLD held-out, cả tám mô hình vượt gấp đôi ngân sách ở mọi mức còn control thì không; ghi `table_review_fa_budget.csv` và `table_review_fa_budget_models.csv`.

In [6]:
# Step 4: false-alarm budgets, 21 address features. Per model the mean over the five splits; the table gives the range over the eight continuous-score models and the number of models above the budget
rowsb = []
for b in BUDGETS:
    for setting in ['holdout', 'control']:
        d = res[(res.budget == b) & (res.test_set == 'out') & (res.setting == setting) & res.model.isin(CONT)].groupby('model').false_alarm_rate.mean()
        d_in = res[(res.budget == b) & (res.test_set == 'in') & (res.setting == setting) & res.model.isin(CONT)].groupby('model').false_alarm_rate.mean() if setting == 'holdout' else None
        d_val = res[(res.budget == b) & (res.test_set == 'valid') & (res.setting == setting) & res.model.isin(CONT)].groupby('model').false_alarm_rate.mean()
        rowsb.append([b, setting, d.min(), d.max(), int((d > b).sum()), int((d > 2 * b).sum()), (d_in.min() if d_in is not None else np.nan), (d_in.max() if d_in is not None else np.nan), d_val.min(), d_val.max()])
fb = pd.DataFrame(rowsb, columns=['budget', 'setting', 'fa_out_min', 'fa_out_max', 'n_models_above_budget', 'n_models_above_twice_budget', 'fa_seen_tld_min', 'fa_seen_tld_max', 'fa_valid_min', 'fa_valid_max'])
fb.to_csv('report/table_review_fa_budget.csv', index=False); print((fb.set_index(['budget', 'setting']) * 1).round(4).to_string())
res[(res.test_set == 'out')].groupby(['model', 'setting', 'budget']).false_alarm_rate.agg(['mean', 'min', 'max']).reset_index().rename(columns={'mean': 'fa_mean5', 'min': 'fa_min5', 'max': 'fa_max5'}).to_csv('report/table_review_fa_budget_models.csv', index=False)      # all ten models, per budget
# all ten models, to show that k-NN and the decision tree (tied scores) are the exception
print(res[(res.test_set == 'out') & (res.setting == 'holdout')].pivot_table(index='model', columns='budget', values='false_alarm_rate').reindex(FAM).round(4).to_string())

                fa_out_min  fa_out_max  n_models_above_budget  n_models_above_twice_budget  fa_seen_tld_min  fa_seen_tld_max  fa_valid_min  fa_valid_max
budget setting                                                                                                                                          
0.005  holdout      0.0151      0.1435                      8                            8           0.0043           0.0058        0.0051        0.0051
       control      0.0052      0.0086                      8                            0              NaN              NaN        0.0050        0.0052
0.010  holdout      0.0405      0.2457                      8                            8           0.0101           0.0115        0.0100        0.0102
       control      0.0099      0.0184                      7                            0              NaN              NaN        0.0101        0.0101
0.020  holdout      0.0850      0.3725                      8                     

## Bước 5. Đối chiếu với Notebook B

- **Làm gì:** so các hàng của lần chia 0 (21 đặc trưng địa chỉ, ngân sách 1%) với `table_rq3_unseen_tld.csv`.
- **Vì sao:** chứng minh notebook này lặp lại đúng phép thử của Notebook B, không phải một phép thử khác.
- **Kiểm tra:** dòng `ok: split 0 reproduces table_rq3_unseen_tld.csv`; chênh lệch lớn nhất nhỏ hơn 0,005. XGBoost có thể lệch nhẹ giữa các máy: trên máy tôi chạy thử chênh 0,0042.

In [7]:
# Step 5: check. Split 0 is the split of Notebook B, so its rows must reproduce the unseen-TLD table of Notebook B
old = pd.read_csv('report/table_rq3_unseen_tld.csv'); old = old[old.features == 'address only, no similarity index'].set_index('model').reindex(FAM)
new = res[(res.split == 0) & (res.setting == 'holdout') & (res.budget == 0.01)].pivot_table(index='model', columns='test_set', values=['caught', 'false_alarm_rate']).reindex(FAM)
d = pd.DataFrame({'fa_unseen': new[('false_alarm_rate', 'out')] - old.false_alarms_unseen_tld, 'caught_unseen': new[('caught', 'out')] - old.caught_unseen_tld,
                  'fa_seen': new[('false_alarm_rate', 'in')] - old.false_alarms_seen_tld, 'caught_seen': new[('caught', 'in')] - old.caught_seen_tld})
print('largest absolute difference to Notebook B on split 0:', float(d.abs().max().max()))
assert float(d.abs().max().max()) < 5e-3, 'split 0 does not reproduce Notebook B'
print('ok: split 0 reproduces table_rq3_unseen_tld.csv')

largest absolute difference to Notebook B on split 0: 1.1102230246251565e-16
ok: split 0 reproduces table_rq3_unseen_tld.csv
